In [ ]:
# pip install transformers datasets torch scikit-learn plotly
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS, TSNE
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import shortest_path
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ripser

### Dataset Selection

Set `DATASET` below to `'imdb'` (binary sentiment) or `'ag_news'` (4-class topic classification).
Everything downstream — data loading, the classifier head size, checkpoint filenames, and the
activation / TDA visualizations — adapts automatically to whichever dataset is selected.

In [ ]:
# ============================================================
# DATASET SELECTION — change this single flag to switch tasks
# ============================================================
DATASET = 'imdb'   # 'imdb' (binary sentiment) or 'ag_news' (4-class topic)
MODEL = 'mini'  # uncased or mini

if MODEL == 'uncased':
    DATASET_CONFIG = {
        'imdb': {
            'display_name':    'IMDB Movie Reviews',
            'num_labels':      2,
            'class_names':     ['Negative', 'Positive'],
            'checkpoint_path': 'checkpoint-uncased-IMDB.pt',
            'max_samples':     None,    # use the full dataset
            'model_path': ".\\bert-base-uncased",
            'layers': 12,
        },
        'ag_news': {
            'display_name':    'AG News',
            'num_labels':      4,
            'class_names':     ['World', 'Sports', 'Business', 'Sci/Tech'],
            'checkpoint_path': 'checkpoint-uncased-agnews.pt',
            'max_samples':     20000,   # cap for feasible local training; set to None for the full ~127k rows
            'model_path': ".\\bert-base-uncased",
            'layers': 12,
        },
    }

elif MODEL == 'mini':
    DATASET_CONFIG = {
        'imdb': {
            'display_name':    'IMDB Movie Reviews',
            'num_labels':      2,
            'class_names':     ['Negative', 'Positive'],
            'checkpoint_path': 'checkpoint_mini_IMDB.pt',
            'max_samples':     None,    # use the full dataset
            'model_path': ".\\bert-mini",
            'layers': 4,
        },
        'ag_news': {
            'display_name':    'AG News',
            'num_labels':      4,
            'class_names':     ['World', 'Sports', 'Business', 'Sci/Tech'],
            'checkpoint_path': 'checkpoint_mini_agnews.pt',
            'max_samples':     20000,   # cap for feasible local training; set to None for the full ~127k rows
            'model_path': ".\\bert-mini",
            'layers': 4,
        },
    }

cfg             = DATASET_CONFIG[DATASET]
NUM_LABELS      = cfg['num_labels']
CLASS_NAMES     = cfg['class_names']
CHECKPOINT_PATH = cfg['checkpoint_path']
MODEL_PATH      = cfg['model_path']
LAYERS          = cfg['layers']    

print(f"Selected dataset: {cfg['display_name']}  | {MODEL_PATH}  |  classes: {CLASS_NAMES}  |  checkpoint: {CHECKPOINT_PATH}")


### Load Dataset

Loads the raw dataset selected above: the IMDB CSV file, or AG News via the `datasets` library.

In [ ]:
if DATASET == 'imdb':
    try:
        df = pd.read_csv('movie_reviews.csv')
        print("Dataset loaded successfully.")
        display(df.head())
    except FileNotFoundError:
        print("Error: 'movie_reviews.csv' not found. Please upload the file or provide the correct path.")

elif DATASET == 'ag_news':
    def load_ag_csv(path):
        # Handles both common AG News CSV layouts:
        #  - Kaggle-style, with header row "Class Index,Title,Description"
        #  - Classic Zhang et al. distribution, no header row, same 3 columns
        raw = pd.read_csv(path)
        expected_cols = {'Class Index', 'Title', 'Description'}
        if not expected_cols.issubset(set(raw.columns)):
            raw = pd.read_csv(path, header=None, names=['Class Index', 'Title', 'Description'])
        return raw.rename(columns={
            'Class Index': 'label', 'Title': 'title', 'Description': 'description'
        })

    train_raw = load_ag_csv('ag_news/train.csv')
    test_raw  = load_ag_csv('ag_news/test.csv')

    ag_df = pd.concat([train_raw, test_raw], ignore_index=True)
    ag_df['text'] = ag_df['title'].astype(str) + '. ' + ag_df['description'].astype(str)
    ag_df['label'] = ag_df['label'].astype(int) - 1   # 1-indexed -> 0-indexed

    df = ag_df[['text', 'label']]

else:
    raise ValueError(f"Unknown DATASET: '{DATASET}'. Choose 'imdb' or 'ag_news'.")

In [ ]:
# Normalize column names / labels so the rest of the pipeline is dataset-agnostic:
# after this cell, df always has a 'review' text column and an integer 'label' column.
if DATASET == 'imdb':
    df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
    # 'review' column already exists in movie_reviews.csv

elif DATASET == 'ag_news':
    df = df.rename(columns={'text': 'review'})
    # 'label' is already an integer 0-3

# Optional subsampling — AG News (~127k rows) is much larger than IMDB (~50k);
# cap it here if training the full set locally would be too slow.
if cfg.get('max_samples') is not None and len(df) > cfg['max_samples']:
    df = df.sample(n=cfg['max_samples'], random_state=42).reset_index(drop=True)
    print(f"Subsampled to {len(df)} rows (max_samples={cfg['max_samples']})")

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df['label']
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Classes ({NUM_LABELS}): {CLASS_NAMES}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

class TextClassificationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts  = dataframe['review'].tolist()
        self.labels = dataframe['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextClassificationDataset(train_df, tokenizer)
val_dataset   = TextClassificationDataset(val_df,   tokenizer)
test_dataset  = TextClassificationDataset(test_df,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)
test_loader  = DataLoader(test_dataset,  batch_size=16)

In [ ]:
class BertClassifierWithActivations(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        # output_hidden_states=True is the key flag —
        # it returns embeddings + one tensor per encoder layer
        self.bert = BertModel.from_pretrained(
            MODEL_PATH,
            local_files_only=True,
            output_hidden_states=True
        )
        hidden_size = self.bert.config.hidden_size  # 256 for bert-mini

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # outputs.hidden_states: tuple of 5 tensors for bert-mini
        #   [0]   → token embedding layer (before any transformer block)
        #   [1-4] → output of each of the 4 encoder layers
        # Each tensor shape: (batch_size, seq_len, 256)
        hidden_states = outputs.hidden_states

        # CLS token at the final layer → used for classification
        cls_final = hidden_states[-1][:, 0, :]

        logits = self.classifier(self.dropout(cls_final))

        return {
            'logits':        logits,
            'hidden_states': hidden_states   # expose everything
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BertClassifierWithActivations(num_labels=NUM_LABELS).to(device)
print(f"Model initialized with {NUM_LABELS} output classes on {device}.")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct = 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            out    = model(input_ids, attention_mask)
            loss   = loss_fn(out['logits'], labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            correct    += (out['logits'].argmax(dim=1) == labels).sum().item()

    n = len(loader.dataset)
    return total_loss / len(loader), correct / n

for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)
    print(f"Epoch {epoch+1} | "
          f"Train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"Val loss {vl_loss:.3f} acc {vl_acc:.3f}")

In [ ]:
# Optional: load a previously saved checkpoint for the SELECTED dataset
# (resumes training state, or just restores weights for inference/extraction).
try:
    # checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Loaded checkpoint '{CHECKPOINT_PATH}' "
          f"(resuming from epoch {start_epoch}, val_acc={checkpoint.get('val_acc', float('nan')):.3f})")
except FileNotFoundError:
    print(f"No checkpoint found at '{CHECKPOINT_PATH}'. Train the model first, then save it below.")

In [ ]:
# ONLY EXECUTE WHEN SAVING MODEL
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'val_acc': vl_acc,
    'val_loss': vl_loss,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Saved checkpoint to '{CHECKPOINT_PATH}'")

In [ ]:
from torch.utils.data import DataLoader, SubsetRandomSampler
import numpy as np

def extract_activations(loader, n_samples=None, seed=42):
    '''
    Returns two dicts keyed by layer index (0 to LAYERS):
      cls_by_layer  : CLS token at each layer
      mean_by_layer : mean-pooled tokens at each layer
    Also returns:
      all_labels  : (N,) — ground-truth labels
      pred_labels : (N,) — model-predicted labels
    seed controls which samples are drawn when n_samples is set.
    '''
    # If subsampling, build a fresh shuffled loader from the same dataset
    if n_samples is not None:
        rng     = np.random.default_rng(seed)
        n_total = len(loader.dataset)
        indices = rng.choice(n_total, size=min(n_samples, n_total), replace=False)
        sampler = SubsetRandomSampler(indices.tolist())
        loader  = DataLoader(
            loader.dataset,
            batch_size=loader.batch_size,
            sampler=sampler,
        )

    model.eval()
    cls_by_layer  = {l: [] for l in range(LAYERS + 1)}
    mean_by_layer = {l: [] for l in range(LAYERS + 1)}
    all_labels    = []
    pred_labels   = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label']

            out           = model(input_ids, attention_mask)
            hidden_states = out['hidden_states']
            logits        = out['logits']

            preds = logits.argmax(dim=1).cpu().numpy()
            pred_labels.append(preds)

            for l, hs in enumerate(hidden_states):
                cls_by_layer[l].append(hs[:, 0, :].cpu().numpy())
                mask     = attention_mask.unsqueeze(-1).float()
                mean_emb = (hs * mask).sum(dim=1) / mask.sum(dim=1)
                mean_by_layer[l].append(mean_emb.cpu().numpy())

            all_labels.append(labels.numpy())

    cls_by_layer  = {l: np.concatenate(v) for l, v in cls_by_layer.items()}
    mean_by_layer = {l: np.concatenate(v) for l, v in mean_by_layer.items()}
    all_labels    = np.concatenate(all_labels)
    pred_labels   = np.concatenate(pred_labels)

    return cls_by_layer, mean_by_layer, all_labels, pred_labels


# Change seed to get a different random subset each time
cls_acts, mean_acts, labels, preds = extract_activations(
    test_loader, n_samples=750, seed=42
)

print("Layers available:", list(cls_acts.keys()))
print("Shape per layer: ", cls_acts[0].shape)
print("Labels shape:    ", labels.shape)
correct_mask              = (labels == preds)
positive_mask             = (labels == 1)
negative_mask             = (labels == 0)
positive_and_correct_mask = correct_mask & positive_mask
negative_and_correct_mask = correct_mask & negative_mask
print(f"Accuracy on extracted subset: {correct_mask.mean():.3f}")
print(f"Positive: {positive_mask.sum()} | Negative: {negative_mask.sum()}")

In [ ]:
def to_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

activations_np = {l: to_numpy(v) for l, v in cls_acts.items()}
labels_np = to_numpy(labels).astype(int)

# Generic N-class masks (works for binary IMDB and 4-class AG News alike)
class_masks = {c: (labels_np == c) for c in range(NUM_LABELS)}

# Color palette — first two entries keep the familiar red/green scheme for the
# binary IMDB case; extra qualitative colors are added for datasets with more classes.
PALETTE = ['#1f77b4', "#fce300",'red', 'green', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
CLASS_COLORS = PALETTE[:NUM_LABELS]

layers = sorted(activations_np.keys())
print(f"Samples: {len(labels_np)}")
for c in range(NUM_LABELS):
    print(f"  {CLASS_NAMES[c]}: {class_masks[c].sum()}")
for l in layers:
    print(f"Layer {l}: {activations_np[l].shape}")

In [ ]:
N_COMPONENTS = 2        # 2 or 3
FILTER_MODE  = 'correct_only'      # 'all' or 'correct_only'
layers = range(1, LAYERS+1)
print([i for i in layers])

In [ ]:
def project_nd(X, method, n_components, random_state=42):
    n = X.shape[0]
    if method == 'pca':
        reducer = PCA(n_components=n_components, random_state=random_state)
    elif method == 'mds':
        reducer = MDS(n_components=n_components, random_state=random_state,
                       n_init=4, n_jobs=-1)
    elif method == 'tsne':
        perplexity = min(30, max(5, (n - 1) // 3))
        reducer = TSNE(n_components=n_components, random_state=random_state,
                        perplexity=perplexity, init='pca')
    else:
        raise ValueError(method)
    return reducer.fit_transform(X)

methods = ['pca', 'mds', 'tsne']
projections = {m: {} for m in methods}

for method in methods:
    print(f"\nComputing {method.upper()} ({N_COMPONENTS}D)...")
    for l in layers:
        print(f"  Layer {l}...")
        projections[method][l] = project_nd(activations_np[l], method, N_COMPONENTS)

print("\nDone.")

In [ ]:
PALETTE = ['#1f77b4', "#ffa500", 'red', 'green', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
CLASS_COLORS = PALETTE[:NUM_LABELS]

# Different marker shapes for colorblind-friendly distinction
CLASS_SYMBOLS = ['x', 'diamond']


def axis_limits(coords, pad=0.08):
    mins, maxs = coords.min(axis=0), coords.max(axis=0)
    pad_amt = (maxs - mins) * pad
    pad_amt[pad_amt == 0] = 1e-3
    return mins - pad_amt, maxs + pad_amt


def plot_layer_evolution_interactive(proj_by_layer, method_name, layers, n_components,
                                      class_masks, correct_mask, class_names, class_colors,
                                      filter_mode='all',
                                      point_size=4, opacity=0.7):
    if filter_mode == 'correct_only':
        show_mask = correct_mask
    elif filter_mode == 'all':
        show_mask = np.ones_like(correct_mask, dtype=bool)
    else:
        raise ValueError("filter_mode must be 'all' or 'correct_only'")

    n_classes = len(class_names)
    n_cols    = 1
    n_layers  = len(layers)
    is_3d     = (n_components == 3)

    specs  = [[{'type': 'scene' if is_3d else 'xy'}] for _ in range(n_layers)]
    titles = [f"Layer {l} — Overlay" for l in layers]

    fig = make_subplots(
        rows=n_layers, cols=n_cols,
        specs=specs,
        subplot_titles=titles,
        vertical_spacing=min(0.04, 1 / max(n_layers - 1, 1))
    )

    ScatterCls = go.Scatter3d if is_3d else go.Scatter

    def make_trace(pts, color, symbol, name, showlegend):
        kwargs = dict(
            x=pts[:, 0],
            y=pts[:, 1],
            mode='markers',
            marker=dict(
                size=point_size,
                color=color,
                opacity=opacity,
                symbol=symbol,
                line=dict(width=0)
            ),
            name=name,
            legendgroup=name,
            showlegend=showlegend,
        )

        if is_3d:
            kwargs['z'] = pts[:, 2]

        return ScatterCls(**kwargs)

    for row_idx, l in enumerate(layers, start=1):
        coords = proj_by_layer[l]
        lo, hi = axis_limits(coords)

        pts_per_class = [
            coords[class_masks[c] & show_mask]
            for c in range(n_classes)
        ]

        # Overlay — all classes in one panel
        for c in range(n_classes):
            symbol = CLASS_SYMBOLS[c] if c < len(CLASS_SYMBOLS) else 'circle'

            fig.add_trace(
                make_trace(
                    pts_per_class[c],
                    class_colors[c],
                    symbol,
                    class_names[c],
                    row_idx == 1
                ),
                row=row_idx,
                col=1
            )

        if is_3d:
            fig.update_scenes(
                xaxis_range=[lo[0], hi[0]],
                xaxis_title='Dim 1',
                yaxis_range=[lo[1], hi[1]],
                yaxis_title='Dim 2',
                zaxis_range=[lo[2], hi[2]],
                zaxis_title='Dim 3',

                # Light gray plotting area
                bgcolor='#f2f2f2',

                row=row_idx,
                col=1
            )
        else:
            fig.update_xaxes(
                range=[lo[0], hi[0]],
                title_text='Dim 1',

                # Light gray plotting area
                showgrid=True,
                gridcolor='white',

                row=row_idx,
                col=1
            )

            fig.update_yaxes(
                range=[lo[1], hi[1]],
                title_text='Dim 2',

                # Light gray plotting area
                showgrid=True,
                gridcolor='white',

                row=row_idx,
                col=1
            )

    filt_label = "All Samples" if filter_mode == 'all' else "Correctly Classified Only"

    fig.update_layout(
        height=380 * n_layers,
        width=700,
        title_text=f"{method_name.upper()} — layer-wise {n_components}D projection ({filt_label})",
        showlegend=True,

        # Overall figure stays white
        paper_bgcolor='white',

        # 2D plotting area is light gray
        plot_bgcolor='#f2f2f2',
    )

    fig.show()
    return fig


for method in methods:
    plot_layer_evolution_interactive(
        projections[method], method, layers, N_COMPONENTS,
        class_masks, correct_mask, CLASS_NAMES, CLASS_COLORS,
        filter_mode=FILTER_MODE
    )

In [ ]:
# Check how many PCA dims you actually need per layer
for l in layers:
    pca = PCA().fit(activations_np[l])
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n90 = np.searchsorted(cumvar, 0.90) + 1
    n95 = np.searchsorted(cumvar, 0.95) + 1
    print(f"Layer {l}: 90% variance in {n90} dims, 95% in {n95} dims")

In [ ]:
# ── 1. PCA — Cumulative explained variance ───────────────────────────────────
# x = number of components, y = % variance captured (per layer)
# Horizontal reference lines at 90% and 95% let you read off the
# minimum dims needed for any desired threshold.

fig_pca = go.Figure()
n90_per_layer, n95_per_layer = {}, {}

for l in layers:
    pca    = PCA().fit(activations_np[l])
    cumvar = np.cumsum(pca.explained_variance_ratio_) * 100   # → percentage
    dims   = np.arange(1, len(cumvar) + 1)

    n90_per_layer[l] = int(np.searchsorted(cumvar, 90)) + 1
    n95_per_layer[l] = int(np.searchsorted(cumvar, 95)) + 1

    fig_pca.add_trace(go.Scatter(
        x=dims, y=cumvar,
        mode='lines+markers', name=f'Layer {l}',
        marker=dict(size=4),
    ))

fig_pca.add_hline(y=90, line_dash='dash', line_color='gray',
                   annotation_text='90%', annotation_position='right')
fig_pca.add_hline(y=95, line_dash='dot',  line_color='gray',
                   annotation_text='95%', annotation_position='right')

fig_pca.update_layout(
    title='PCA — Cumulative Explained Variance per Layer',
    xaxis_title='Number of PCA components',
    yaxis_title='Cumulative explained variance (%)',
    yaxis=dict(range=[0, 101]),
    width=800, height=500, legend_title='Layer',
)
fig_pca.show()

print("PCA dims needed per layer:")
for l in layers:
    print(f"  Layer {l}: 90% → {n90_per_layer[l]} dims | 95% → {n95_per_layer[l]} dims")

MAX_PCA_DIMS = max(n95_per_layer.values())
print(f"\nRecommended N_PCA_DIMS (max 95% threshold across all layers): {MAX_PCA_DIMS}")

In [ ]:
# ── 2. MDS — Normalised stress vs number of dimensions ───────────────────────
# The closest MDS analog to the PCA variance plot.
# Kruskal's normalised stress measures how faithfully an n-dim embedding
# preserves the original pairwise distances (0 = perfect, ~0.2 = poor).
# The "elbow" in the stress curve is the MDS equivalent of the 90/95% threshold.
#
# NOTE: this runs MDS once per (layer, n_dims) pair — computationally heavier
# than PCA. MAX_MDS_DIMS=15 and n_init=1 keep it tractable; raise if you have time.

MAX_MDS_DIMS = 15
MDS_DIMS     = list(range(1, MAX_MDS_DIMS + 1))

def kruskal_stress(X, n_components):
    '''Normalised stress — scale-independent, range roughly [0, 1].'''
    mds = MDS(n_components=n_components, random_state=42, n_init=1,
               max_iter=200, n_jobs=-1, normalized_stress=True)
    mds.fit(X)
    return mds.stress_

print("Computing MDS stress curves (this may take a few minutes)...")
fig_mds = go.Figure()

for l in layers:
    stresses = []
    for d in MDS_DIMS:
        s = kruskal_stress(activations_np[l], d)
        stresses.append(s)
        print(f"  Layer {l}, {d}D → stress {s:.4f}")

    fig_mds.add_trace(go.Scatter(
        x=MDS_DIMS, y=stresses,
        mode='lines+markers', name=f'Layer {l}',
        marker=dict(size=5),
    ))

# Kruskal's rule-of-thumb thresholds
fig_mds.add_hline(y=0.10, line_dash='dash', line_color='green',
                   annotation_text='Good (0.10)', annotation_position='right')
fig_mds.add_hline(y=0.20, line_dash='dot',  line_color='orange',
                   annotation_text='Acceptable (0.20)', annotation_position='right')

fig_mds.update_layout(
    title='MDS — Normalised Stress vs Number of Dimensions per Layer',
    xaxis_title='Number of MDS dimensions',
    yaxis_title="Kruskal's normalised stress",
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    width=800, height=500, legend_title='Layer',
)
fig_mds.show()

In [ ]:
X = activations_np[4]   # last layer, as example
X_pca7  = PCA(n_components=7).fit_transform(X)

D_full  = pairwise_distances(X)
D_pca7  = pairwise_distances(X_pca7)

# Normalised stress of PCA-7 reduction
stress_pca7 = np.sqrt(
    np.sum((D_full - D_pca7)**2) / np.sum(D_full**2)
)
print(f"PCA-7 stress on layer 4: {stress_pca7:.4f}")
# Expect: close to the ~0.08 you saw from MDS-7 on the stress curve

# PERSISTENT HOMOLOGY

In [ ]:
K_NEIGHBORS = 2    # k for the k-NN graph
N_PCA_DIMS  = 3   # PCA target dimensionality
N_MDS_DIMS  = 7    # MDS target dimensionality
MAX_DIM     = 2    # compute H0 and H1 only
CORRECT_ONLY = True
FILTRATION_MODE = 'vr'
NORMALIZE_DISTANCES = True 
POSITIVE_OR_NEGATIVE = 'all'

# Select mask
if POSITIVE_OR_NEGATIVE == 'positive':
    sample_mask = (
        positive_and_correct_mask if CORRECT_ONLY
        else positive_mask
    )
    filter_label = (
        "correctly classified positive samples"
        if CORRECT_ONLY else "all positive samples"
    )

elif POSITIVE_OR_NEGATIVE == 'negative':
    sample_mask = (
        negative_and_correct_mask if CORRECT_ONLY
        else negative_mask
    )
    filter_label = (
        "correctly classified negative samples"
        if CORRECT_ONLY else "all negative samples"
    )

elif POSITIVE_OR_NEGATIVE == 'all':
    sample_mask = (
        correct_mask if CORRECT_ONLY
        else np.ones(len(labels), dtype=bool)
    )
    filter_label = (
        "correctly classified samples"
        if CORRECT_ONLY else "all samples"
    )

else:
    raise ValueError("POSITIVE_OR_NEGATIVE must be one of {'positive', 'negative', 'all'}")

print(f"Samples used: {sample_mask.sum()} / {len(sample_mask)} ({filter_label})")
print(f"Filtration mode: {FILTRATION_MODE}")

In [ ]:
print(f"Computing PCA-{N_PCA_DIMS} projections...")
pca_proj = {}
for l in layers:
    pca_proj[l] = PCA(n_components=N_PCA_DIMS, random_state=42).fit_transform(activations_np[l])
    print(f"  Layer {l}: {pca_proj[l].shape}")

"""print(f"\nComputing MDS-{N_MDS_DIMS} projections...")
mds_proj = {}
for l in layers:
    mds = MDS(n_components=N_MDS_DIMS, random_state=42,
               n_init=4, n_jobs=-1, normalized_stress=True)
    mds_proj[l] = mds.fit_transform(activations_np[l])
    print(f"  Layer {l}: {mds_proj[l].shape}")"""

In [ ]:
def build_knn_dist_matrix(X, k):
    graph = kneighbors_graph(X, n_neighbors=k, mode='connectivity',
                              include_self=False)
    graph = (graph + graph.T)
    graph.data[:] = 1.0
    dist = shortest_path(graph, method='D', directed=False)
    max_finite = dist[np.isfinite(dist)].max() if np.any(np.isfinite(dist)) else 1.0
    dist[~np.isfinite(dist)] = max_finite + 1.0
    return dist


def build_vr_dist_matrix(X):
    from sklearn.metrics import pairwise_distances
    return pairwise_distances(X, metric='euclidean')


def normalize_dist_matrix(D):
    """Divide by max pairwise distance → all distances in [0, 1]."""
    max_val = D[np.triu_indices_from(D, k=1)].max()
    return D / max_val if max_val > 0 else D


def get_dist_matrix(X, mode):
    if mode == 'knn':
        D = build_knn_dist_matrix(X, K_NEIGHBORS)
    elif mode == 'vr':
        D = build_vr_dist_matrix(X)
    else:
        raise ValueError(f"FILTRATION_MODE must be 'knn' or 'vr', got '{mode}'")

    if NORMALIZE_DISTANCES and mode == 'vr':
        D = normalize_dist_matrix(D)     # k-NN already scale-free (hop counts)

    return D


def run_ripser(dist_mat):
    return ripser.ripser(dist_mat, distance_matrix=True, maxdim=MAX_DIM)['dgms']


print(f"\nBuilding distance matrices and computing persistence...")
results = {'pca': {}, 'mds': {}}

"""for l in layers:
    for key, proj in [('pca', pca_proj[l]), ('mds', mds_proj[l])]:
        proj_filtered = proj[sample_mask]
        dist_mat      = get_dist_matrix(proj_filtered, FILTRATION_MODE)
        dgms          = run_ripser(dist_mat)
        results[key][l] = dgms
        print(f"  Layer {l} [{key.upper():>3}]:  "
              f"H0={len(dgms[0]):>3} bars   H1={len(dgms[1]):>3} bars")"""
        
for l in layers:
    for key, proj in [('pca', pca_proj[l])]:
        proj_filtered = proj[sample_mask]
        dist_mat      = get_dist_matrix(proj_filtered, FILTRATION_MODE)
        dgms          = run_ripser(dist_mat)
        results[key][l] = dgms
        print(f"  Layer {l} [{key.upper():>3}]:  "
              f"H0={len(dgms[0]):>3} bars   H1={len(dgms[1]):>3} bars   H2={len(dgms[2]):>3} bars")

In [ ]:
def _barcode_ax(ax, dgm, color, title):
    """Horizontal bar chart for one homology dimension."""
    finite = dgm[np.isfinite(dgm[:, 1])]
    infbars = dgm[~np.isfinite(dgm[:, 1])]

    x_max = finite[:, 1].max() * 1.15 if len(finite) else 1.0

    if len(infbars):
        x_max = max(x_max, infbars[:, 0].max() * 1.3)

    y = 0

    # Finite bars, longest first
    for b, d in sorted(
        finite,
        key=lambda r: r[1] - r[0],
        reverse=True
    ):
        ax.plot(
            [b, d],
            [y, y],
            color=color,
            lw=1.4,
            alpha=0.75,
            solid_capstyle='butt'
        )
        y += 1

    # Essential/infinite bars
    for b, _ in infbars:
        ax.plot(
            [b, x_max],
            [y, y],
            color=color,
            lw=1.4,
            alpha=0.75,
            linestyle='--',
            solid_capstyle='butt'
        )
        y += 1

    ax.set_xlim(-0.1, x_max)
    ax.set_ylim(-1, max(y + 1, 2))
    ax.set_xlabel('Filtration value', fontsize=8)
    ax.set_yticks([])
    ax.set_title(title, fontsize=9)


def _diagram_ax(ax, dgms, title):
    """Persistence diagram showing H0, H1 and H2 on one axis."""

    palette = [
        'tab:blue',    # H0
        'tab:orange',  # H1
        'tab:green',   # H2
    ]

    all_finite = []

    # Plot H0, H1 and H2
    for dim, dgm in enumerate(dgms[:MAX_DIM + 1]):
        col = palette[dim]

        finite = dgm[np.isfinite(dgm[:, 1])]
        infbars = dgm[~np.isfinite(dgm[:, 1])]

        if len(finite):
            ax.scatter(
                finite[:, 0],
                finite[:, 1],
                c=col,
                s=12,
                alpha=0.7,
                label=f'H{dim}',
                zorder=3
            )
            all_finite.append(finite)

        if len(infbars):
            # Essential classes plotted as triangles above
            # the finite persistence points.
            if all_finite:
                y_ess = np.concatenate(all_finite)[:, 1].max() * 1.15
            else:
                y_ess = 2.0

            ax.scatter(
                infbars[:, 0],
                [y_ess] * len(infbars),
                c=col,
                s=14,
                marker='^',
                alpha=0.7,
                zorder=3
            )

    if all_finite:
        vals = np.concatenate(all_finite)

        lo = vals.min() - 0.05
        hi = vals.max() * 1.15

        # Diagonal
        ax.plot(
            [lo, hi],
            [lo, hi],
            'k--',
            lw=0.8,
            alpha=0.4
        )

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

    ax.set_xlabel('Birth', fontsize=8)
    ax.set_ylabel('Death', fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc='lower right')


def total_persistence(dgm):
    """Sum of lifetimes (death − birth) over all finite bars."""
    finite = dgm[np.isfinite(dgm[:, 1])]

    if len(finite):
        return float(
            np.sum(finite[:, 1] - finite[:, 0])
        )

    return 0.0

In [ ]:
n_layers = len(layers)

METHOD_META = [
    ('pca', f'PCA-{N_PCA_DIMS}', 'tab:blue', 'tab:cyan')
]

for key, label, col_h0, col_h1 in METHOD_META:

    # H0 barcode | H1 barcode | H2 barcode | Persistence diagram
    fig, axes = plt.subplots(
        n_layers,
        4,
        figsize=(20, 4 * (n_layers)),
        squeeze=False,
    )

    fig.suptitle(
        f'{label} | k-NN graph (k={K_NEIGHBORS}) | '
        f'Barcodes and Persistence Diagrams',
        fontsize=13,
        y=1.01
    )

    for row, l in enumerate(layers):

        dgms = results[key][l]

        # ----------------------------------------------------------
        # H0
        # ----------------------------------------------------------
        _barcode_ax(
            axes[row, 0],
            dgms[0],
            'tab:blue',
            f'Layer {l} — H0 barcode'
        )

        # ----------------------------------------------------------
        # H1
        # ----------------------------------------------------------
        _barcode_ax(
            axes[row, 1],
            dgms[1],
            'tab:orange',
            f'Layer {l} — H1 barcode'
        )

        # ----------------------------------------------------------
        # H2
        # ----------------------------------------------------------
        _barcode_ax(
            axes[row, 2],
            dgms[2],
            'tab:green',
            f'Layer {l} — H2 barcode'
        )

        # ----------------------------------------------------------
        # Persistence diagram: H0 + H1 + H2
        # ----------------------------------------------------------
        _diagram_ax(
            axes[row, 3],
            dgms,
            f'Layer {l} — Persistence diagram'
        )

    # Keep common filtration scale for all barcode panels
    for ax in axes[:, 0]:
        ax.set_xlim(0.0, 1.4)

    for ax in axes[:, 1]:
        ax.set_xlim(0.0, 1.4)

    for ax in axes[:, 2]:
        ax.set_xlim(0.0, 1.4)

    plt.tight_layout()
    plt.show()

In [ ]:
# Build dynamic title components
subset_label = filter_label  # already created when selecting the mask

filtration_label = {
    "vr": "Vietoris–Rips",
    "knn": f"{K_NEIGHBORS}-NN Graph",
}.get(FILTRATION_MODE, FILTRATION_MODE)

distance_label = "Normalized" if NORMALIZE_DISTANCES else "Raw"

method_label = f"PCA-{N_PCA_DIMS} vs MDS-{N_MDS_DIMS}"


# ── compute totals ────────────────────────────────────────────────────────────

# Persistence for each homology dimension
tp = {
    key: {
        dim: [
            total_persistence(results[key][l][dim])
            for l in layers
        ]
        for dim in range(MAX_DIM + 1)
    }
    for key, *_ in METHOD_META
}

# Total persistence summed across H0 + H1 + H2
tp_total = {
    key: [
        sum(
            total_persistence(results[key][l][dim])
            for dim in range(MAX_DIM + 1)
        )
        for l in layers
    ]
    for key, *_ in METHOD_META
}


# ── plot ──────────────────────────────────────────────────────────────────────

# One subplot for H0, H1, H2 + one for the sum
n_plots = MAX_DIM + 2

fig_tp = make_subplots(
    rows=1,
    cols=n_plots,
    subplot_titles=[
        *[
            f'H{d} — Total Persistence'
            for d in range(MAX_DIM + 1)
        ],
        'H0 + H1 + H2 — Total Persistence'
    ]
)


styles = {
    'pca': dict(dash='solid', color='royalblue'),
    'mds': dict(dash='dash', color='tomato'),
}


for key, label, *_ in METHOD_META:

    # ----------------------------------------------------------
    # H0, H1, H2
    # ----------------------------------------------------------
    for dim in range(MAX_DIM + 1):

        fig_tp.add_trace(
            go.Scatter(
                x=list(layers),
                y=tp[key][dim],
                mode='lines+markers',
                name=label,
                legendgroup=label,
                showlegend=(dim == 0),
                line=dict(**styles[key]),
                marker=dict(size=8),
            ),
            row=1,
            col=dim + 1
        )

    # ----------------------------------------------------------
    # H0 + H1 + H2
    # ----------------------------------------------------------
    fig_tp.add_trace(
        go.Scatter(
            x=list(layers),
            y=tp_total[key],
            mode='lines+markers',
            name=f'{label} — H0+H1+H2',
            legendgroup=label,
            showlegend=False,
            line=dict(
                **styles[key],
                width=3,
            ),
            marker=dict(size=8),
        ),
        row=1,
        col=n_plots
    )


# ── axes ──────────────────────────────────────────────────────────────────────

fig_tp.update_xaxes(
    title_text='Layer',
    tickmode='linear',
    tick0=0,
    dtick=1,
)

fig_tp.update_yaxes(
    title_text='Total Persistence',
)


# ── layout ────────────────────────────────────────────────────────────────────

fig_tp.update_layout(
    title=dict(
        text=(
            f"Total Persistence per Layer<br>"
            f"{'3-PCA'} | {filtration_label} | {distance_label} distances<br>"
        ),
        x=0.5,
        xanchor="center",
        yanchor="top",
    ),

    margin=dict(t=120),

    height=420,
    width=1200,

    legend=dict(
        x=0.01,
        y=0.99,
    ),
)

fig_tp.show()

In [ ]:
from sklearn.metrics import pairwise_distances
import plotly.graph_objects as go

scale_stats = {}
for l in layers:
    for key, proj in [('pca', pca_proj[l]), ('mds', mds_proj[l])]:
        D = pairwise_distances(proj[sample_mask])
        upper = D[np.triu_indices_from(D, k=1)]
        scale_stats[(key, l)] = {
            'median': np.median(upper),
            'mean':   np.mean(upper),
            'std':    np.std(upper),
        }

fig_scale = go.Figure()
for key, label, color in [('pca', f'PCA-{N_PCA_DIMS}', 'royalblue'),
                           ('mds', f'MDS-{N_MDS_DIMS}', 'tomato')]:
    fig_scale.add_trace(go.Scatter(
        x=list(layers),
        y=[scale_stats[(key, l)]['median'] for l in layers],
        mode='lines+markers',
        name=f'{label} — median dist',
        line=dict(color=color),
        marker=dict(size=8),
    ))

fig_scale.update_layout(
    title='Median pairwise distance per layer '
          '(scale diagnostic — if this grows, VR total persistence grows trivially)',
    xaxis_title='Layer', yaxis_title='Median pairwise distance',
    xaxis=dict(tickmode='linear'), height=400, width=750,
)
fig_scale.show()

## LINKING NUMBER

In [ ]:
import networkx as nx
import itertools
from sklearn.neighbors import NearestNeighbors

# ── switches ──────────────────────────────────────────────────────────────────
K_LINK        = 8     # k for k-NN graph (paper recommends 6–15)
EPS_PCT       = 70    # ε = this percentile of 1st-NN distances
N_SUB         = 4     # midpoint quadrature subdivisions per edge
MIN_CYCLE_LEN = 4     # discard shorter cycles (likely graph artefacts)
MAX_CYCLES    = 60    # cap β per class (shortest cycles first)
N_LINK_SAMP   = 300   # subsample per class before graph construction

# ── graph construction (Algorithm 1 step ii) ─────────────────────────────────
def build_filtered_knn_graph(X, k, eps_pct):
    """ε-filtered k-NN graph with ε = eps_pct-th percentile of 1-NN distances."""
    n = len(X)
    nbrs = NearestNeighbors(n_neighbors=min(k + 1, n)).fit(X)
    dists, inds = nbrs.kneighbors(X)
    eps = np.percentile(dists[:, 1], eps_pct)   # 1-NN distances for threshold

    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j, d in zip(inds[i, 1:], dists[i, 1:]):
            if d <= eps:
                G.add_edge(i, int(j))
    return G


# ── fundamental cycle basis (Algorithm 1 step iii / Definition H.1) ──────────
def extract_fundamental_cycles(G, X_3d, min_len=4, max_cycles=60):
    """
    Spanning-forest cycle basis.
    Each non-tree edge (u,v) yields the fundamental cycle:
    unique tree path u→v closed by the edge v→u.
    Returns list of (n_i, 3) coordinate arrays, shortest cycles first.
    """
    cycles = []
    for comp in nx.connected_components(G):
        if len(comp) < 3:
            continue
        sub  = G.subgraph(comp).copy()
        root = next(iter(comp))
        T    = nx.bfs_tree(sub, root).to_undirected()

        tree_edges = frozenset(frozenset(e) for e in T.edges())
        non_tree   = [e for e in sub.edges() if frozenset(e) not in tree_edges]

        for u, v in non_tree:
            try:
                path = nx.shortest_path(T, u, v)  # unique path in tree
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue
            if len(path) >= min_len:
                cycles.append(path)

    cycles.sort(key=len)          # shortest = geometrically tightest
    cycles = cycles[:max_cycles]
    return [X_3d[np.array(c)] for c in cycles]


# ── Gauss linking integral (Algorithm 1 step iv / Appendix H.3) ──────────────
def gauss_link(C1, C2, n_sub=4):
    """
    Midpoint-quadrature Gauss linking integral for two PL closed curves in R³.

      link(C1,C2) = 1/(4π) ∮_C1 ∮_C2  (x-y)·(dx×dy) / |x-y|³

    C1: (n1,3), C2: (n2,3) — ordered vertices; curve closes last→first.
    Returns a float; round to nearest integer to get the linking number.
    """
    # Edge tangent vectors
    T1 = np.roll(C1, -1, axis=0) - C1    # (n1, 3)
    T2 = np.roll(C2, -1, axis=0) - C2    # (n2, 3)

    # Midpoints of n_sub equal subdivisions of [0,1]
    fracs = (np.arange(n_sub) + 0.5) / n_sub   # (n_sub,)

    # Quadrature sample points on each edge: (n_edges, n_sub, 3)
    X_pts = C1[:, None, :] + fracs[None, :, None] * T1[:, None, :]
    Y_pts = C2[:, None, :] + fracs[None, :, None] * T2[:, None, :]

    # T1[i] × T2[j]  →  (n1, n2, 3)  — constant per edge pair
    crosses = np.cross(T1[:, None, :], T2[None, :, :])

    total = 0.0
    for si in range(n_sub):
        x = X_pts[:, si, :]                         # (n1, 3)
        for tj in range(n_sub):
            y    = Y_pts[:, tj, :]                  # (n2, 3)
            diff = x[:, None, :] - y[None, :, :]   # (n1, n2, 3)
            d3   = np.sum(diff**2, axis=-1) ** 1.5  # (n1, n2)
            dots = np.sum(diff * crosses, axis=-1)   # (n1, n2)
            with np.errstate(divide='ignore', invalid='ignore'):
                total += np.where(d3 > 1e-20, dots / d3, 0.0).sum()

    return total / (4.0 * np.pi * n_sub**2)


# ── per-layer linking computation ─────────────────────────────────────────────
def compute_linking_for_layer(X_all, mask_a, mask_b,
                               k=8, eps_pct=70, n_sub=4,
                               min_cycle=4, max_cycles=60,
                               n_samp=200, seed=42):
    """
    Compute linking summary between two class clouds at one layer.
    Subsampling is done here; PCA-3 projection is fitted on the full X_all
    so both classes share the same coordinate system.
    """
    rng = np.random.default_rng(seed)

    # Project full layer activations to R³ (Algorithm 1 step i)
    X_3d = PCA(n_components=3, random_state=seed).fit_transform(X_all)

    def subsample(mask):
        idx = np.where(mask)[0]
        return rng.choice(idx, min(n_samp, len(idx)), replace=False)

    idx_a, idx_b = subsample(mask_a), subsample(mask_b)
    Xa, Xb = X_3d[idx_a], X_3d[idx_b]

    Ga = build_filtered_knn_graph(Xa, k, eps_pct)
    Gb = build_filtered_knn_graph(Xb, k, eps_pct)

    Ca = extract_fundamental_cycles(Ga, Xa, min_cycle, max_cycles)
    Cb = extract_fundamental_cycles(Gb, Xb, min_cycle, max_cycles)

    if not Ca or not Cb:
        return dict(sum_abs=0.0, max_abs=0.0, n_pairs=0,
                    n_linked=0, beta_a=len(Ca), beta_b=len(Cb))

    link_vals = np.array([gauss_link(c1, c2, n_sub) for c1 in Ca for c2 in Cb])
    rounded   = np.abs(np.round(link_vals))

    return dict(
        sum_abs  = float(rounded.sum()),
        max_abs  = float(rounded.max()),
        n_pairs  = len(link_vals),
        n_linked = int((rounded >= 1).sum()),
        beta_a   = len(Ca),
        beta_b   = len(Cb),
    )

In [ ]:
# Class masks intersected with the correctness filter
class_masks_link = {
    c: sample_mask  & (labels_np == c) for c in range(NUM_LABELS)
}

# All unordered class pairs  — 1 for IMDB, 6 for AG News
class_pairs = list(itertools.combinations(range(NUM_LABELS), 2))

print(f"Computing Gauss linking numbers:")
print(f"  {len(class_pairs)} class pair(s) x {len(layers)} layers")
print(f"  k={K_LINK}, ε={EPS_PCT}th pct, N_sub={N_SUB}, "
      f"max_β={MAX_CYCLES}, n_samp={N_LINK_SAMP}\n")

link_results = {pair: {} for pair in class_pairs}

for l in layers:
    X_layer = activations_np[l]     # (N, d) — cls or mean-pooled activations
    for pair in class_pairs:
        ca, cb = pair
        res = compute_linking_for_layer(
            X_layer,
            class_masks_link[ca],
            class_masks_link[cb],
            k=K_LINK, eps_pct=EPS_PCT, n_sub=N_SUB,
            min_cycle=MIN_CYCLE_LEN, max_cycles=MAX_CYCLES,
            n_samp=N_LINK_SAMP,
        )
        link_results[pair][l] = res
        print(f"  Layer {l:>2}  ({CLASS_NAMES[ca]} vs {CLASS_NAMES[cb]}): "
              f"β=({res['beta_a']:>3},{res['beta_b']:>3})  "
              f"pairs={res['n_pairs']:>5}  linked={res['n_linked']:>4}  "
              f"Σ|lk|={res['sum_abs']:>6.1f}  max|lk|={res['max_abs']:.0f}")

In [ ]:
plot_layers = [l for l in layers if l != 0]   # exclude embedding layer

PAIR_COLORS = ['royalblue', 'tomato', 'seagreen',
               'darkorange', 'mediumpurple', 'saddlebrown']

METRICS = [
    ('sum_abs',  'Σ|lk| — total linking',   'Total Linking  Σ|lk|'),
    ('max_abs',  'max|lk| — strongest link', 'Max |lk|'),
    ('n_linked', 'Linked pairs (|lk| ≥ 1)', 'Linked Pairs'),
]

fig_link = make_subplots(
    rows=1, cols=len(METRICS),
    subplot_titles=[m[2] for m in METRICS],
    horizontal_spacing=0.10,
)

for pi, (pair, color) in enumerate(zip(class_pairs, PAIR_COLORS)):
    ca, cb = pair
    lbl    = f'{CLASS_NAMES[ca]} vs {CLASS_NAMES[cb]}'

    for mi, (key, ytitle, _) in enumerate(METRICS):
        fig_link.add_trace(
            go.Scatter(
                x=plot_layers,
                y=[link_results[pair][l][key] for l in plot_layers],
                mode='lines+markers',
                name=lbl,
                legendgroup=lbl,
                showlegend=(mi == 0),
                line=dict(color=color, width=2),
                marker=dict(size=8),
            ),
            row=1, col=mi + 1,
        )

fig_link.update_xaxes(title_text='Layer', tickmode='linear',
                       tick0=plot_layers[0], dtick=1)
for mi, (_, ytitle, _) in enumerate(METRICS):
    fig_link.update_yaxes(title_text=ytitle, row=1, col=mi + 1)

fig_link.add_hline(y=0, line_dash='dash', line_color='lightgrey',
                    line_width=1, row=1, col=1)

filter_str = "correctly classified" if CORRECT_ONLY else "all samples"
fig_link.update_layout(
    title=dict(
        text=(
            f"Gauss Linking Numbers per Layer<br>"
            f"k={K_LINK}-NN | ε={EPS_PCT}th pct | "
            f"max β={MAX_CYCLES} | N_sub={N_SUB}<br>"
        ),
        x=0.5, xanchor='center', yanchor='top',
    ),
    margin=dict(t=130),
    height=450, width=1100,
    legend=dict(x=1.01, y=0.99, xanchor='left'),
)
fig_link.show()